# SegResNet CV Fold MPS Inference

Runs local inference for the five fold-specific `dice_focal_fold_X` SegResNet checkpoints. For each fold `X`, the notebook selects the corresponding patient rows from `data/cv_splits_qc.csv`, loads `notebooks/output/models/dice_focal_fold_X/best_metric_model.pth`, and writes masks under `notebooks/output/models/dice_focal_fold_X/masks`.

In [1]:
import os
import sys
from pathlib import Path

os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")
os.environ.setdefault("MPLCONFIGDIR", "/private/tmp/mpl-cache")
os.environ.setdefault("XDG_CACHE_HOME", "/private/tmp/xdg-cache")

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")
print(f"Python: {sys.executable}")

Project root: /Users/ricca/Desktop/Health_Informatics_Internship_2026
Python: /opt/miniconda3/bin/python


In [2]:
import torch

print(f"torch: {torch.__version__}")
print(f"MPS built: {torch.backends.mps.is_built()}")
print(f"MPS available: {torch.backends.mps.is_available()}")

if not torch.backends.mps.is_available():
    raise RuntimeError(
        "MPS is not available in this notebook kernel. Select a Python/conda "
        "environment where torch.backends.mps.is_available() is True."
    )

device = torch.device("mps")
print(f"Using device: {device}")

torch: 2.11.0
MPS built: True
MPS available: True
Using device: mps


In [3]:
import importlib
import pandas as pd

from scripts import run_local_fold_minus1_inference

run_local_fold_minus1_inference = importlib.reload(run_local_fold_minus1_inference)

from scripts.run_local_fold_minus1_inference import (
    filter_readable_cases,
    resolve_project_path,
    select_cases_from_split,
)

FOLD_IDS = [0, 1, 2, 3, 4]
DEVICE = "mps"
INPUT_DATA = "notebooks/nifti_data"
SPLIT_CSV = "data/cv_splits_qc.csv"
CONFIG_PATH = "config/train_config.yaml"
CHECKPOINT_TEMPLATE = "notebooks/output/models/dice_focal_fold_{fold}/best_metric_model.pth"
OUTPUT_DIR_TEMPLATE = "notebooks/output/models/dice_focal_fold_{fold}/masks"
MAX_CASES = None
SKIP_EXISTING = True

fold_rows = []
for fold in FOLD_IDS:
    checkpoint_path = resolve_project_path(CHECKPOINT_TEMPLATE.format(fold=fold))
    output_dir = resolve_project_path(OUTPUT_DIR_TEMPLATE.format(fold=fold))
    cases = select_cases_from_split(INPUT_DATA, SPLIT_CSV, test_fold=fold)
    if MAX_CASES is not None:
        cases = cases[: int(MAX_CASES)]
    readable_cases, skipped_cases = filter_readable_cases(cases)
    fold_rows.append(
        {
            "fold": fold,
            "checkpoint_exists": checkpoint_path.exists(),
            "checkpoint_path": str(checkpoint_path),
            "output_dir": str(output_dir),
            "selected_cases": len(cases),
            "readable_cases": len(readable_cases),
            "unreadable_cases": len(skipped_cases),
            "patient_ids": ", ".join(case.patient_id for case in readable_cases),
        }
    )

fold_plan = pd.DataFrame(fold_rows)
display(fold_plan)

missing_checkpoints = fold_plan.loc[~fold_plan["checkpoint_exists"], "checkpoint_path"].tolist()
if missing_checkpoints:
    raise FileNotFoundError("Missing checkpoint(s):\n" + "\n".join(missing_checkpoints))

,fold,checkpoint_exists,checkpoint_path,output_dir,selected_cases,readable_cases,unreadable_cases,patient_ids
0,0,True,/Users/ricca/Desktop/Health_Informatics_Intern...,/Users/ricca/Desktop/Health_Informatics_Intern...,28,28,0,"TAVI_007, TAVI_029, TAVI_032, TAVI_038, TAVI_0..."
1,1,True,/Users/ricca/Desktop/Health_Informatics_Intern...,/Users/ricca/Desktop/Health_Informatics_Intern...,28,28,0,"TAVI_014, TAVI_042, TAVI_049, TAVI_055, TAVI_0..."
2,2,True,/Users/ricca/Desktop/Health_Informatics_Intern...,/Users/ricca/Desktop/Health_Informatics_Intern...,28,28,0,"TAVI_023, TAVI_026, TAVI_041, TAVI_060, TAVI_0..."
3,3,True,/Users/ricca/Desktop/Health_Informatics_Intern...,/Users/ricca/Desktop/Health_Informatics_Intern...,28,28,0,"TAVI_015, TAVI_024, TAVI_031, TAVI_048, TAVI_0..."
4,4,True,/Users/ricca/Desktop/Health_Informatics_Intern...,/Users/ricca/Desktop/Health_Informatics_Intern...,28,28,0,"TAVI_004, TAVI_016, TAVI_017, TAVI_020, TAVI_0..."


In [4]:
for fold in FOLD_IDS:
    checkpoint = CHECKPOINT_TEMPLATE.format(fold=fold)
    output_dir = OUTPUT_DIR_TEMPLATE.format(fold=fold)
    argv = [
        "--checkpoint",
        checkpoint,
        "--input_data",
        INPUT_DATA,
        "--split_csv",
        SPLIT_CSV,
        "--output_dir",
        output_dir,
        "--config",
        CONFIG_PATH,
        "--test_fold",
        str(fold),
        "--device",
        DEVICE,
    ]
    if MAX_CASES is not None:
        argv.extend(["--max_cases", str(int(MAX_CASES))])
    if SKIP_EXISTING:
        argv.append("--skip_existing")

    print(f"\n=== Fold {fold} ===")
    print(f"Checkpoint: {resolve_project_path(checkpoint)}")
    print(f"Output dir: {resolve_project_path(output_dir)}")
    run_local_fold_minus1_inference.main(argv)


=== Fold 0 ===
Checkpoint: /Users/ricca/Desktop/Health_Informatics_Internship_2026/notebooks/output/models/dice_focal_fold_0/best_metric_model.pth
Output dir: /Users/ricca/Desktop/Health_Informatics_Internship_2026/notebooks/output/models/dice_focal_fold_0/masks
Checkpoint: /Users/ricca/Desktop/Health_Informatics_Internship_2026/notebooks/output/models/dice_focal_fold_0/best_metric_model.pth
Output dir: /Users/ricca/Desktop/Health_Informatics_Internship_2026/notebooks/output/models/dice_focal_fold_0/masks
Running 28 fold 0 case(s) on device=mps.
Loaded ema_model_state_dict from checkpoint.


/opt/miniconda3/lib/python3.13/site-packages/monai/utils/deprecate_utils.py:320: FutureWarning: monai.transforms.spatial.dictionary Orientationd.__init__:labels: Current default value of argument `labels=(('L', 'R'), ('P', 'A'), ('I', 'S'))` was changed in version None from `labels=(('L', 'R'), ('P', 'A'), ('I', 'S'))` to `labels=None`. Default value changed to None meaning that the transform now uses the 'space' of a meta-tensor, if applicable, to determine appropriate axis labels.
  warn_deprecated(argname, msg, warning_category)


[TAVI_007] inference
[TAVI_007] Dice=0.8254 IoU=0.7026 surfaceHD=6.99mm HD=6.99mm
[TAVI_029] inference
[TAVI_029] Dice=0.9100 IoU=0.8349 surfaceHD=8.62mm HD=8.62mm
[TAVI_032] inference
[TAVI_032] Dice=0.9118 IoU=0.8380 surfaceHD=4.17mm HD=4.02mm
[TAVI_038] inference
[TAVI_038] Dice=0.9200 IoU=0.8519 surfaceHD=4.58mm HD=4.58mm
[TAVI_061] inference
[TAVI_061] Dice=0.8927 IoU=0.8063 surfaceHD=15.10mm HD=15.10mm
[TAVI_064] inference
[TAVI_064] Dice=0.8092 IoU=0.6795 surfaceHD=6.00mm HD=6.00mm
[TAVI_107] inference
[TAVI_107] Dice=0.9413 IoU=0.8891 surfaceHD=3.21mm HD=3.21mm
[TAVI_108] inference
[TAVI_108] Dice=0.9088 IoU=0.8328 surfaceHD=5.60mm HD=5.60mm
[TAVI_117] inference
[TAVI_117] Dice=0.9250 IoU=0.8605 surfaceHD=3.48mm HD=3.48mm
[TAVI_118] inference
[TAVI_118] Dice=0.9005 IoU=0.8191 surfaceHD=6.00mm HD=6.00mm
[TAVI_121] inference
[TAVI_121] Dice=0.9196 IoU=0.8511 surfaceHD=4.22mm HD=4.22mm
[TAVI_133] inference
[TAVI_133] Dice=0.8965 IoU=0.8123 surfaceHD=4.91mm HD=4.91mm
[TAVI_146] inf

In [5]:
summary_frames = []
metric_frames = []

for fold in FOLD_IDS:
    output_dir = resolve_project_path(OUTPUT_DIR_TEMPLATE.format(fold=fold))
    summary_path = output_dir / "fold_minus1_metrics_summary.csv"
    metrics_path = output_dir / "fold_minus1_metrics.csv"
    if summary_path.exists():
        summary = pd.read_csv(summary_path)
        summary.insert(0, "fold", fold)
        summary_frames.append(summary)
    if metrics_path.exists():
        metrics = pd.read_csv(metrics_path)
        metrics.insert(0, "inference_fold", fold)
        metric_frames.append(metrics)

all_summaries = pd.concat(summary_frames, ignore_index=True) if summary_frames else pd.DataFrame()
all_metrics = pd.concat(metric_frames, ignore_index=True) if metric_frames else pd.DataFrame()

display(all_summaries)
display(all_metrics.sort_values(["inference_fold", "patient_id", "mask_type"]) if not all_metrics.empty else all_metrics)

,fold,mask_type,case_count,mean_dice,median_dice,mean_surface_hausdorff_mm,median_surface_hausdorff_mm,mean_surface_hausdorff95_mm,median_surface_hausdorff95_mm,mean_hausdorff_mm,...,mean_iou,median_iou,mean_pred_volume_ml,median_pred_volume_ml,mean_ground_truth_volume_ml,median_ground_truth_volume_ml,mean_volume_difference_ml,median_volume_difference_ml,mean_volume_ratio_pred_to_gt,median_volume_ratio_pred_to_gt
0,0,postprocessed,28,0.895703,0.908199,5.717023,4.985271,2.766570,3.0,5.686250,...,0.812796,0.831837,135.302720,127.035014,126.967309,118.537945,8.335411,5.899706,1.067465,1.038357
1,0,raw,28,0.895399,0.907109,20.267177,5.079058,2.776948,3.0,20.236405,...,0.812309,0.830009,135.396283,127.035014,126.967309,118.537945,8.428974,5.899706,1.068205,1.038357
2,1,postprocessed,28,0.891181,0.895176,6.357666,6.086759,3.002925,3.0,6.351932,...,0.805038,0.810253,137.796105,136.053058,133.604199,128.265165,4.191906,3.681739,1.033543,1.025073
3,1,raw,28,0.890201,0.895176,14.682096,6.235682,3.055360,3.0,14.676362,...,0.803512,0.810253,138.170315,136.053058,133.604199,128.265165,4.566116,4.037472,1.035793,1.026811
4,2,postprocessed,28,0.884629,0.891464,6.387708,5.903454,3.017445,3.0,6.358666,...,0.794713,0.804187,134.822006,133.925021,130.182588,128.076775,4.639418,3.522882,1.042662,1.035861
5,2,raw,28,0.884499,0.890072,9.318985,5.949450,3.017445,3.0,9.289944,...,0.794504,0.801919,134.863831,133.925021,130.182588,128.076775,4.681243,3.522882,1.042949,1.035861
6,3,postprocessed,28,0.891160,0.896718,5.812319,5.491245,2.836419,3.0,5.753795,...,0.805030,0.812773,130.712253,124.553218,123.944041,122.329978,6.768212,6.256056,1.063069,1.054245
7,3,raw,28,0.890046,0.896718,18.003576,5.491245,3.105241,3.0,17.945052,...,0.803477,0.812773,131.253955,124.553218,123.944041,122.329978,7.309914,6.256056,1.065893,1.060301
8,4,postprocessed,28,0.875385,0.891191,6.681618,6.000000,3.348187,3.0,6.627259,...,0.781561,0.803741,143.362160,133.729385,143.205883,128.323233,0.156277,0.890148,1.006724,1.006860
9,4,raw,28,0.873669,0.891191,17.153667,6.005553,5.292524,3.0,17.099307,...,0.778989,0.803741,144.392619,133.729385,143.205883,128.323233,1.186736,0.897531,1.010922,1.008245


,inference_fold,patient_id,fold,mask_type,mask_path,dice,surface_hausdorff_mm,surface_hausdorff95_mm,hausdorff_mm,hausdorff95_mm,iou,pred_volume_ml,ground_truth_volume_ml,volume_difference_ml,volume_ratio_pred_to_gt,pred_voxels,ground_truth_voxels
0,0,TAVI_007,0,postprocessed,/Users/ricca/Desktop/Health_Informatics_Intern...,0.825351,6.985800,3.086729,6.985800,2.209754,0.702637,114.262850,100.025178,14.237672,1.142341,288601,252640
1,0,TAVI_007,0,raw,/Users/ricca/Desktop/Health_Informatics_Intern...,0.825351,6.985800,3.086729,6.985800,2.209754,0.702637,114.262850,100.025178,14.237672,1.142341,288601,252640
2,0,TAVI_029,0,postprocessed,/Users/ricca/Desktop/Health_Informatics_Intern...,0.910000,8.624580,2.791698,8.624580,0.882812,0.834863,140.944345,138.547820,2.396526,1.017297,241129,237029
3,0,TAVI_029,0,raw,/Users/ricca/Desktop/Health_Informatics_Intern...,0.906210,117.787193,3.000000,117.787193,0.882812,0.828505,142.113382,138.547820,3.565562,1.025735,243129,237029
4,0,TAVI_032,0,postprocessed,/Users/ricca/Desktop/Health_Informatics_Intern...,0.911850,4.172698,2.900243,4.015760,0.683594,0.837982,93.853433,91.065052,2.788382,1.030620,267789,259833
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
275,4,TAVI_358,4,raw,/Users/ricca/Desktop/Health_Informatics_Intern...,0.874015,5.410155,3.000000,5.176664,1.491553,0.776223,105.098909,100.310175,4.788734,1.047739,283447,270532
276,4,TAVI_365,4,postprocessed,/Users/ricca/Desktop/Health_Informatics_Intern...,0.673070,19.500000,10.500000,19.500000,9.000000,0.507239,115.276343,115.818404,-0.542061,0.995320,563982,566634
277,4,TAVI_365,4,raw,/Users/ricca/Desktop/Health_Informatics_Intern...,0.673070,19.500000,10.500000,19.500000,9.000000,0.507239,115.276343,115.818404,-0.542061,0.995320,563982,566634
278,4,TAVI_372,4,postprocessed,/Users/ricca/Desktop/Health_Informatics_Intern...,0.896029,6.224039,3.000000,6.224039,1.026320,0.811642,184.354391,173.148400,11.205991,1.064719,291700,273969
